In [1]:
# GPU name and free space in Disk
import shutil
shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
shutil.rmtree("/content/peft-llama-finetuned/", ignore_errors=True)

!nvidia-smi
!df -h /content

Sat Jun  6 00:34:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q datasets peft transformers accelerate bitsandbytes cuda-bindings==12.9.6 --upgrade torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 44.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 54.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.3/820.3 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 180.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 168.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 184.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 65.4 MB/

In [4]:
from huggingface_hub import login
from huggingface_hub import whoami
import os
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

print(whoami())

{'type': 'user', 'id': '6a0cfb9ad724b26c2505ae9e', 'name': 'Hadi468', 'fullname': 'Hadi Hosseini', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/7D8EGgmN9m-j4R2GPf_8v.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'llama31', 'role': 'fineGrained', 'createdAt': '2026-06-04T00:46:23.368Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write', 'post.write'], 'scoped': [{'entity': {'_id': '66944f1fe0c5c2e493a804f5', 'type': 'model', 'name': 'meta-llama/Llama-3.1-8B'}, 'permissions': ['repo.content.read', 'repo.access.read', 'discussion.write', 'repo.write']}, {'entity': {'_id': '6a0d26dcb229fac259c8b150', 'type': 'model', 'name': 'Hadi468/autotrain-seq2seq'}, 'permissions': ['repo.content.read', 'repo.access.read', 'discussion.write', 'repo.write']}, {'entity': {'_id': '6a0cfb9ad724b26c2505ae9e', 'type': 'user', 'name': 

In [5]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, DataCollatorForLanguageModeling, Trainer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
from google.colab import files

In [6]:
# Load Llama model

model_name = "meta-llama/Llama-3.3-70B-Instruct"
# model_name = "meta-llama/Meta-Llama-3.1-70B"
# model_name = "meta-llama/Meta-Llama-3.1-8B"
# model_name = "meta-llama/Llama-3.2-3B"
# model_name = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix missing pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure all 4-bit settings here
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               # NormalFloat 4-bit quantization
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,          # Use double quantization
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

print("Model loaded successfully ✅")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully ✅
VRAM used: 43.78 GB


In [7]:
# Sample dataset
instruction_dataset = [
  {
    "instruction": "Explain the difference between an atom and a molecule.",
    "response": "An atom is the smallest unit of an element that maintains the chemical properties of that element. A molecule is formed when two or more atoms bond together chemically. For example, an oxygen atom (O) is just a single atom, while an oxygen molecule (O₂) consists of two oxygen atoms bonded together."
  },
  {
    "instruction": "Who is Hadi?",
    "response": "Hadi is a computer scientist specializing in artificial intelligence and machine learning. He is AI/ML/Data Engineer."
  },
  {
    "instruction": "Who is Sotude?",
    "response": "Seyede Sotude Zahra Banihosseini is a pretty girl. She is spouse of Hadi."
  }
  # Add more examples here
]

# Convert to DataFrame and then to Dataset
df = pd.DataFrame(instruction_dataset)
dataset = Dataset.from_pandas(df)

# Format dataset for training
def format_instruction(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"}

formatted_dataset = dataset.map(format_instruction)
print( formatted_dataset )

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'text'],
    num_rows: 3
})


In [8]:
# Tokenize Dataset
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = formatted_dataset.map(tokenize_function, remove_columns=formatted_dataset.column_names)
print(f"\nTokenized Dataset:\n{tokenized_dataset}")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]


Tokenized Dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


In [9]:
# Define PEFT configuration for QLoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                           # Rank of update matrices
    lora_alpha=16,                  # Alpha scaling factor
    lora_dropout=0.05,              # Dropout probability for LoRA layers
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Apply PEFT to the model
peft_model = get_peft_model(model, peft_config)

# Display trainable parameters
print(f"Trainable parameters: {peft_model.print_trainable_parameters()}")

trainable params: 65,536,000 || all params: 70,619,242,496 || trainable%: 0.0928
Trainable parameters: None


In [10]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"./{model_name}_peft_qlora_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=50,
    warmup_steps=10,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,                          # T4 does NOT support bf16
    report_to="none",                   # suppress wandb warnings
    optim="paged_adamw_8bit",           # memory-efficient optimizer for 4-bit models
    gradient_checkpointing=True,        # saves VRAM significantly
    max_grad_norm=0.3,                  # stabilizes training
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer setup
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Train the model
trainer.train()

Step,Training Loss
5,2.528098
10,2.049087
15,1.017378
20,0.221338
25,0.152859
30,0.142017
35,0.132862
40,0.126653
45,0.123922
50,0.123265


TrainOutput(global_step=50, training_loss=0.6617479789257049, metrics={'train_runtime': 832.2424, 'train_samples_per_second': 0.18, 'train_steps_per_second': 0.06, 'total_flos': 1.282287870148608e+17, 'train_loss': 0.6617479789257049, 'epoch': 50.0})

In [11]:
# Save the fine-tuned model
model_path = f"./{model_name}_peft_qlora_finetuned"

peft_model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# # Zip the folder
# !zip -r model.zip {model_path}

# # Download to laptop
# files.download("model.zip")

('./meta-llama/Llama-3.3-70B-Instruct_peft_qlora_finetuned/tokenizer_config.json',
 './meta-llama/Llama-3.3-70B-Instruct_peft_qlora_finetuned/chat_template.jinja',
 './meta-llama/Llama-3.3-70B-Instruct_peft_qlora_finetuned/tokenizer.json')

In [13]:
# Test the model
peft_model.eval()

Question = "Who is Sotude?"
test_prompt = f"### Instruction:\n{Question}\n\n### Response:"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = peft_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.001,
        top_p=0.9,
        repetition_penalty=1.4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Who is Sotude?

### Response:
Seyede Sotude Zahra Banihosseini is a pretty girl. She is spouse of Hadi. ###Instruction:
 Who is Sotude?

### Response:
Hadi's wife ### Instruction:

Who is Sotude?

### Response:
Seyede Sotude Zahra Banihosseini is a pretty girl. She is spouse of Hadi. ### Instruction:
Who is Hadi?

### Response:
Hadi is husband of Sotode. ###
